# Sync Results: results_alpha_eps01

This notebook identifies missing results for the `alpha_eps01` experiment group and facilitates downloading them from the remote server.

In [ ]:
import os
import subprocess
from pathlib import Path

remote_host = "snorlax-login"
remote_base_dir = "~/final_research/results"
local_results_dir = "../results_alpha_eps01/results"
experiment_pattern = "weighted_alpha_eps01_v4_*"

def run_command(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
    return result.stdout.strip()

# 1. Get local results
local_results = set(os.listdir(local_results_dir)) if os.path.exists(local_results_dir) else set()
print(f"Found {len(local_results)} results locally.")

# 2. Get remote results
remote_cmd = f"ssh {remote_host} 'ls -d {remote_base_dir}/{experiment_pattern}'"
remote_output = run_command(remote_cmd)
remote_paths = remote_output.split('\n') if remote_output else []
remote_results = {os.path.basename(p) for p in remote_paths if p}
print(f"Found {len(remote_results)} results on remote matching pattern.")

# 3. Identify missing
missing = sorted(list(remote_results - local_results))
print(f"Total missing results: {len(missing)}")
if missing:
    print("First 5 missing:", missing[:5])


## Download Missing Results

We will zip the missing results on the remote server and then download them to maintain structure and efficiency.

In [ ]:
if missing:
    # Create a temporary file on remote with the list of directories to zip
    missing_list_str = "\\n".join([f"results/{m}" for m in missing])
    remote_zip_cmd = (
        f"ssh {remote_host} \"cd ~/final_research && "
        f"echo -e '{missing_list_str}' > missing_list.txt && "
        f"zip -r missing_results_alpha_eps01.zip -@ < missing_list.txt && "
        f"rm missing_list.txt\""
    )
    
    print("Zipping missing results on remote...")
    print(run_command(remote_zip_cmd))
    
    print("Downloading zip file...")
    download_cmd = f"scp {remote_host}:~/final_research/missing_results_alpha_eps01.zip ../"
    print(run_command(download_cmd))
    
    print("Download complete. You can now unzip missing_results_alpha_eps01.zip into results_alpha_eps01/results/")
else:
    print("No missing results to download.")
